# DocFinder — Colab Pipeline v3 (anti-idle + OCR + Qwen A/B)

Pipeline d'indexation GPU + serveur `/encode` + `/rerank` + (optionnel) `/encode_qwen`.

## Pré-requis (à faire AVANT de lancer les cellules)

1. **Runtime GPU** : Exécution → Modifier le type d'exécution → T4 (ou L4).
2. **Tunnels Cloudflare vivants** :
   - Tunnel #1 (Mac → `docfinder.jinkohub.digital`) : lancé via `launchctl` sur le Mac (LaunchAgent `digital.jinkohub.docfinder`). Doit renvoyer 200 sur `/health`.
   - Tunnel #2 (Colab → `encode.jinkohub.digital`) : token à coller cellule 7.
3. **Mac uvicorn up** : `launchctl list | grep docfinder` retourne un PID actif.

## Ordre d'exécution

1 → 2 → 3 → **(restart runtime)** → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

⚠️ Après le `pip install` (cellule 3), **Exécution → Redémarrer la session** puis reprendre cellule 2.

## Anti-idle / anti-shutdown

Colab gratuit a deux shutdowns :

- **Idle de tab** (~90 min navigateur inactif) → utilise le **hack JS cellule 5** (F12 → Console → paste).
- **Cell appears hung** (pas de stdout pendant X min) → la cellule 9 print toutes les 50s via `keepalive` thread daemon.

Les deux ensemble tiennent une session `normale` 12 h (limite dure runtime).


## Cellule 1 — Cloner / mettre à jour le repo


In [ ]:
import os, subprocess

BRANCH = 'claude/review-project-cloudflare-ggcYD'
REPO = 'https://github.com/kofekod23/docfinder.git'

if not os.path.exists('/content/docfinder/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/content/docfinder'], check=True)
else:
    subprocess.run(['git', '-C', '/content/docfinder', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/docfinder', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/docfinder', 'reset', '--hard', f'origin/{BRANCH}'], check=True)

head = subprocess.check_output(['git', '-C', '/content/docfinder', 'rev-parse', '--short', 'HEAD'], text=True).strip()
msg = subprocess.check_output(['git', '-C', '/content/docfinder', 'log', '-1', '--pretty=%s'], text=True).strip()
print(f'✅ HEAD → {head}: {msg}')


## Cellule 2 — Installer les dépendances

⚠️ Après cette cellule : **Exécution → Redémarrer la session** puis reprendre cellule 1 (ou 3 si rien n'a changé dans le repo).

`transformers>=4.51` requis pour Qwen3-Embedding (L#25 lessons).


In [ ]:
!pip install -q \
  FlagEmbedding==1.3.5 \
  'transformers>=4.51,<5' \
  'sentence-transformers>=3.3' \
  'fastapi>=0.110' \
  'uvicorn[standard]>=0.27' \
  httpx pydantic easyocr PyMuPDF python-docx python-pptx openpyxl
print('✅ deps installées. Restart runtime si nouvelles versions (Exécution → Redémarrer la session).')


## Cellule 3 — Installer cloudflared + lancer le tunnel #2

Ce tunnel expose `localhost:8001` (serveur `/encode` + `/rerank` + `/encode_qwen`) vers `encode.jinkohub.digital`.

Colle le token du tunnel #2 dans `TUNNEL2_TOKEN`.


In [ ]:
TUNNEL2_TOKEN = ''  # ← colle ici le token du 2e tunnel (cloudflared tunnel run --token ...)

import subprocess, os, time

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-qO', '/usr/local/bin/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# Idempotent : kill ancien tunnel avant d'en relancer un
subprocess.run(['pkill', '-f', 'cloudflared.*tunnel.*run'], check=False)
time.sleep(1)

assert TUNNEL2_TOKEN.strip(), 'TUNNEL2_TOKEN vide — colle le token du 2e tunnel Cloudflare'

log = open('/content/cloudflared.log', 'wb')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', TUNNEL2_TOKEN],
    stdout=log, stderr=log,
)
time.sleep(3)
print(f'✅ cloudflared PID {proc.pid} — log: /content/cloudflared.log')


## Cellule 4 — Variables d'environnement

À renseigner (mêmes valeurs que le `.env` du Mac) :

- `COLAB_QUERY_TOKEN` — secret partagé pour `/encode`.
- `CF_ACCESS_CLIENT_ID` / `CF_ACCESS_CLIENT_SECRET` — service token Cloudflare Access.

### Flags optionnels

- `DOCFINDER_FORCE_REINDEX=1` : ignore le checkpoint mtime et ré-encode tous les docs. Utile après un fix d'extraction (OCR etc.). **+30 min à 2 h** sur GPU.
- `DOCFINDER_ENABLE_RERANKER=1` : charge `bge-reranker-v2-m3` pour `/rerank` (défaut ON).
- `DOCFINDER_ENABLE_QWEN=1` : charge `Qwen3-Embedding-0.6B` pour `/encode_qwen` (A/B Plan 2).
- `DOCFINDER_COLLECTION` : nom de la collection Qdrant côté Mac (défaut `docfinder_v2`).


In [ ]:
import os

# Obligatoire
os.environ['MAC_BASE_URL']            = 'https://docfinder.jinkohub.digital'
os.environ['DOCFINDER_ROOT']          = '/Users/julien/Documents'
os.environ['COLAB_QUERY_TOKEN']       = ''  # ← valeur .env Mac
os.environ['CF_ACCESS_CLIENT_ID']     = ''  # ← CF Access service token ID
os.environ['CF_ACCESS_CLIENT_SECRET'] = ''  # ← CF Access service token secret

# Flags
os.environ['DOCFINDER_FORCE_REINDEX']   = '0'  # '1' pour re-index complet après fix extraction
os.environ['DOCFINDER_ENABLE_RERANKER'] = '1'  # '0' pour désactiver /rerank
os.environ['DOCFINDER_ENABLE_QWEN']     = '0'  # '1' pour charger Qwen (A/B Plan 2)
os.environ['DOCFINDER_COLLECTION']      = 'docfinder_v2'

for k in ('COLAB_QUERY_TOKEN', 'CF_ACCESS_CLIENT_ID', 'CF_ACCESS_CLIENT_SECRET'):
    assert os.environ[k].strip(), f'{k} est vide — remplis-le avant de lancer'
print('✅ env OK — flags :',
      'FORCE_REINDEX=' + os.environ['DOCFINDER_FORCE_REINDEX'],
      'RERANKER=' + os.environ['DOCFINDER_ENABLE_RERANKER'],
      'QWEN=' + os.environ['DOCFINDER_ENABLE_QWEN'])


## Cellule 5 — Anti-idle navigateur (hack JS)

⚠️ Colab déconnecte le runtime après ~90 min sans interaction dans l'onglet. Le hack ci-dessous simule un clic sur le bouton de connexion toutes les 60 s.

**À faire UNE fois après avoir lancé ce notebook** :

1. Ouvre les DevTools de Chrome/Safari (`F12` ou `Cmd+Alt+I`).
2. Va dans l'onglet **Console**.
3. Paste le code ci-dessous, appuie Entrée.

```javascript
function ClickConnect(){
    var btn = document.querySelector('colab-toolbar-button#connect');
    if (btn) { btn.click(); console.log('keepalive click', new Date().toLocaleTimeString()); }
    var btn2 = document.querySelector('colab-connect-button');
    if (btn2) { btn2.click(); console.log('keepalive click2', new Date().toLocaleTimeString()); }
}
setInterval(ClickConnect, 60000);
console.log('✅ keepalive JS lancé (click toutes les 60s)');
```

Tant que l'onglet est ouvert (même en arrière-plan), le runtime reste connecté.


## Cellule 6 — Pré-warm navigateur (option)

Si tu lances avec `DOCFINDER_FORCE_REINDEX=1` et que le Mac héberge les docs sur Google Drive FS, certains `/files/raw` renvoient 502 (deadlock File Provider Extension). La pipeline retry auto mais ça ralentit.

**Workaround** (à faire une fois dans un Terminal Mac avant) :

```bash
find ~/Documents -type f \( -name '*.pdf' -o -name '*.docx' -o -name '*.pptx' -o -name '*.xlsx' \) \
  -not -name '~$*' -exec head -c 1 {} \; > /dev/null 2>&1
```

Cette boucle matérialise ~400 fichiers Google Drive localement (~5-10 min). Ensuite tous les `/files/raw` passent en 200.

Cellule facultative côté Colab (sans effet direct mais utile comme rappel).


## Cellule 7 — Smoke test Mac (facultatif)

Vérifie que le Mac répond avant de lancer la pipeline (évite d'attendre 30 s de retry si Mac down).


In [ ]:
import requests, os

headers = {
    'CF-Access-Client-Id': os.environ['CF_ACCESS_CLIENT_ID'],
    'CF-Access-Client-Secret': os.environ['CF_ACCESS_CLIENT_SECRET'],
    'User-Agent': 'DocFinder/1.0',
}

for path in ('/health', '/files/list?root=/Users/julien/Documents&limit=1'):
    url = os.environ['MAC_BASE_URL'] + path
    try:
        r = requests.get(url, headers=headers, timeout=15)
        print(f'{path:60} → {r.status_code}')
    except Exception as e:
        print(f'{path:60} → ERROR {type(e).__name__}: {e}')

print('\nSi /health != 200 → Mac uvicorn down. Vérifie côté Mac :')
print('  launchctl list | grep docfinder')
print('  launchctl kickstart -k gui/$(id -u)/digital.jinkohub.docfinder')


## Cellule 8 — Keepalive stdout (anti-idle côté Python)

Thread daemon qui print l'heure toutes les 50 s → empêche Colab de considérer la cellule suivante comme "hung".


In [ ]:
import threading, time, datetime

_keepalive_stop = threading.Event()

def _heartbeat():
    n = 0
    while not _keepalive_stop.is_set():
        n += 1
        now = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'[keepalive #{n} {now}] runtime actif', flush=True)
        _keepalive_stop.wait(50)

_kt = threading.Thread(target=_heartbeat, name='keepalive', daemon=True)
_kt.start()
print('✅ keepalive Python démarré (print toutes les 50s)')


## Cellule 9 — Lancement unifié (query_server + pipeline)

Exécute `colab_helpers_cell.py` qui charge BGE-M3 (+ optionnel reranker + Qwen), lance uvicorn `/encode` sur :8001 en thread, puis run la pipeline d'indexation avec **auto-retry backoff 15s → 300s** (ne meurt plus sur 502 / timeout Mac transient).

**Cellule bloquante** — tant qu'elle tourne, tout est live. `Stop` sur la cellule = tout s'arrête propre.


In [ ]:
exec(open('/content/docfinder/colab_helpers_cell.py').read())


## Cellule 10 — Status Qdrant (à lancer dans une autre session)

Si tu veux vérifier la progression de l'indexation depuis le Mac :

```bash
curl -s http://localhost:6333/collections/docfinder_v2 | jq '.result | {points_count, vectors_count}'
```

## Mesure A/B finale (après indexation complète)

Sur le Mac, quand la pipeline a terminé (plus de FAIL, juste des retries qui réussissent) :

```bash
cd ~/docfinder
python -m tasks.rerank_ab --mode off --out tasks/ab_out/rerank_off_v2.csv
python -m tasks.rerank_ab --compare tasks/ab_out/rerank_off.csv tasks/ab_out/rerank_off_v2.csv
```

Seuil décision OCR : ΔMRR ≥ +0.02 → merge features.
